# 01 — Introduction à PySpark et la SparkUI

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tanouar/1to1code/blob/main/pySpark/notebooks/01_introduction_jobSpark.ipynb)

Ce notebook crée une **SparkSession**, ouvre la **SparkUI** et exécute 5 tests de complexité croissante.

> Gardez la **SparkUI ouverte dans un onglet à côté** de ce notebook pour observer chaque job en temps réel.


In [ ]:
# Installation + setup (Colab)
!pip install -q pyspark findspark
!git clone --filter=blob:none --sparse https://github.com/tanouar/1to1code.git -q \
  && cd 1to1code && git sparse-checkout set pySpark -q

import os, sys
PYSPARK_TOOLS = os.path.join('/content', '1to1code', 'pySpark')
if PYSPARK_TOOLS not in sys.path:
    sys.path.insert(0, PYSPARK_TOOLS)

from sparktools.colab_setup import setup_colab

spark, monitor, helper = setup_colab("01 - Introduction PySpark")


remote: Enumerating objects: 1, done.
remote: Counting objects: 100% (1/1), done.
remote: Total 1 (delta 0), reused 1 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (1/1), 1.04 KiB | 1.04 MiB/s, done.
remote: Enumerating objects: 13, done.
remote: Counting objects: 100% (13/13), done.
remote: Compressing objects: 100% (11/11), done.
remote: Total 13 (delta 1), reused 10 (delta 1), pack-reused 0 (from 0)
Receiving objects: 100% (13/13), 49.94 KiB | 4.99 MiB/s, done.
Resolving deltas: 100% (1/1), done.
  Preparing metadata (setup.py) ... done
Try `serve_kernel_port_as_iframe` instead. 


<IPython.core.display.Javascript object>

✓ SparkUI ouverte → cliquez sur le lien ci-dessus
✓ Spark 4.0.2  |  App : 01 - Introduction PySpark
✓ Memory : 4g  |  Shuffle partitions : 8
✓ SparkJobMonitor et SparkHelper initialisés


## TEST 1 — Comptage simple

On crée un petit DataFrame et on compte ses lignes.  
→ SparkUI : **1 job** doit apparaître dans l'onglet **Jobs**.


In [5]:
data = [
    ("Alice",   25, "IT",      50000),
    ("Bob",     30, "RH",      45000),
    ("Charlie", 35, "IT",      60000),
    ("David",   28, "Finance", 55000),
]
df = spark.createDataFrame(data, ["nom", "age", "departement", "salaire"])

result = monitor.execute_and_monitor(
    lambda: df.count(),
    "TEST 1: Comptage simple"
)
print(f"Résultat : {result} lignes")



🔵 TEST 1: Comptage simple
📌 Tracking ID: 4c9de955

✅ SUCCESS
⏱️  Durée: 0.62s
📊 Spark Job ID(s): [3, 2]
📦 Stages: 3 | Tasks: 5
🌐 Spark UI: http://localhost:4040/jobs/
Résultat : 4 lignes


## TEST 2 — Agrégation par département

On groupe les données par département.  
→ SparkUI : observez les **2 Stages** dans ce job (Map + Reduce = 1 shuffle).


In [6]:
monitor.execute_and_monitor(
    lambda: df.groupBy("departement").count().show(),
    "TEST 2: Agrégation par département"
)



🔵 TEST 2: Agrégation par département
📌 Tracking ID: 4e5f2f5e
+-----------+-----+
|departement|count|
+-----------+-----+
|         RH|    1|
|         IT|    2|
|    Finance|    1|
+-----------+-----+


✅ SUCCESS
⏱️  Durée: 1.98s
📊 Spark Job ID(s): [5, 4]
📦 Stages: 3 | Tasks: 5
🌐 Spark UI: http://localhost:4040/jobs/


## TEST 3 — Gros volume (10 millions de lignes)

On mesure la vitesse de Spark sur un volume important.  
→ SparkUI : regardez le nombre de **Tasks** exécutées en parallèle.


In [7]:
result = monitor.execute_and_monitor(
    lambda: spark.range(0, 10_000_000).toDF("id").count(),
    "TEST 3: Génération et comptage de 10M lignes"
)
print(f"Résultat : {result:,} lignes")



🔵 TEST 3: Génération et comptage de 10M lignes
📌 Tracking ID: f89a37d4

✅ SUCCESS
⏱️  Durée: 0.89s
📊 Spark Job ID(s): [7, 6]
📦 Stages: 3 | Tasks: 5
🌐 Spark UI: http://localhost:4040/jobs/
Résultat : 10,000,000 lignes


## TEST 4 — Analyse d'un DataFrame avec SparkHelper


In [8]:
from pyspark.sql.functions import rand

df_test = spark.range(0, 1000).toDF("id") \
               .withColumn("valeur", (rand() * 100).cast("int"))

monitor.execute_and_monitor(
    lambda: helper.show_dataframe_info(df_test, "DataFrame Test"),
    "TEST 4: Analyse DataFrame"
)



🔵 TEST 4: Analyse DataFrame
📌 Tracking ID: c35b1082

📊 Informations sur: DataFrame Test

🔢 Nombre de lignes: 1,000
📋 Nombre de colonnes: 2

📝 Schéma:
root
 |-- id: long (nullable = false)
 |-- valeur: integer (nullable = true)


👀 Aperçu des données:
+---+------+
|id |valeur|
+---+------+
|0  |22    |
|1  |51    |
|2  |27    |
|3  |26    |
|4  |51    |
+---+------+
only showing top 5 rows

📈 Statistiques:
+-------+-----------------+------------------+
|summary|               id|            valeur|
+-------+-----------------+------------------+
|  count|             1000|              1000|
|   mean|            499.5|            49.322|
| stddev|288.8194360957494|28.504891595037588|
|    min|                0|                 0|
|    max|              999|                99|
+-------+-----------------+------------------+


✅ SUCCESS
⏱️  Durée: 1.69s
📊 Spark Job ID(s): [12, 11, 10, 9, 8]
📦 Stages: 7 | Tasks: 11
🌐 Spark UI: http://localhost:4040/jobs/


## TEST 5 — Analyse du partitionnement


In [9]:
df_big = spark.range(0, 1_000_000).toDF("id")

monitor.execute_and_monitor(
    lambda: helper.get_partition_info(df_big),
    "TEST 5: Analyse du partitionnement"
)



🔵 TEST 5: Analyse du partitionnement
📌 Tracking ID: b9de669e

🗂️  Informations de Partitionnement

📦 Nombre de partitions: 2

📊 Distribution des lignes par partition:
   Min: 500,000
   Max: 500,000
   Moyenne: 500,000

📋 Détail par partition:
   Partition 0: 500,000 lignes
   Partition 1: 500,000 lignes

✅ SUCCESS
⏱️  Durée: 7.29s
📊 Spark Job ID(s): [13]
📦 Stages: 1 | Tasks: 2
🌐 Spark UI: http://localhost:4040/jobs/


## Historique des jobs

Résumé de tous les jobs exécutés.  
→ SparkUI : vérifiez que les noms correspondent bien à l'onglet **Jobs**.


In [10]:
# ============================================
# HISTORIQUE COMPLET
# ============================================

monitor.show_history()


📜 HISTORIQUE DES JOBS

📊 Statistiques globales:
   Total jobs: 6
   Réussis: 6
   Échoués: 0
   Durée totale: 18.24s


1. TEST 1: Comptage simple
   Tracking ID: 354610d6
   Spark Job IDs: [1, 0]
   Stages: 3 | Tasks: 5
   Durée: 5.77s
   Status: ✅ SUCCESS

2. TEST 1: Comptage simple
   Tracking ID: 4c9de955
   Spark Job IDs: [3, 2]
   Stages: 3 | Tasks: 5
   Durée: 0.62s
   Status: ✅ SUCCESS

3. TEST 2: Agrégation par département
   Tracking ID: 4e5f2f5e
   Spark Job IDs: [5, 4]
   Stages: 3 | Tasks: 5
   Durée: 1.98s
   Status: ✅ SUCCESS

4. TEST 3: Génération et comptage de 10M lignes
   Tracking ID: f89a37d4
   Spark Job IDs: [7, 6]
   Stages: 3 | Tasks: 5
   Durée: 0.89s
   Status: ✅ SUCCESS

5. TEST 4: Analyse DataFrame
   Tracking ID: c35b1082
   Spark Job IDs: [12, 11, 10, 9, 8]
   Stages: 7 | Tasks: 11
   Durée: 1.69s
   Status: ✅ SUCCESS

6. TEST 5: Analyse du partitionnement
   Tracking ID: b9de669e
   Spark Job IDs: [13]
   Stages: 1 | Tasks: 2
   Durée: 7.29s
   Status: ✅

In [11]:
last_job = monitor.get_last_job()
if last_job:
    print(f"Dernier job   : {last_job['name']}")
    print(f"Durée         : {last_job['duration']:.2f}s")
    print(f"Status        : {last_job['status']}")
    print(f"Spark Job IDs : {last_job['spark_job_ids']}")


Dernier job   : TEST 5: Analyse du partitionnement
Durée         : 7.29s
Status        : ✅ SUCCESS
Spark Job IDs : [13]
